# Ollama
- 로컬에서 오픈소스 LLM을 쉽게 실행하고 Serving 할 수 있게 해주는 경량화되니 Runtime program이다.

* **특징**

  * Mac, Windows, Linux에서 동작한다.
  * 한 줄 명령어로 Llama, Mistral, Gemma 등 여러 최신 LLM을 **다운로드하고 바로 실행**할 수 있다.
  * CLI(터미널) 기반이지만, API 서버로도 동작해 **자신만의 ChatGPT**를 구축하거나, 다른 앱과 쉽게 연동할 수 있다.
  * 모델 크기를 자동으로 최적화(quantization 등)해서, 일반 PC에서도 빠르게 돌릴 수 있다.
  * `ollama run llama3.2` 처럼 명령어 한 줄로 모델 구동 가능.

* **주요 활용**

  * 사내/개인 비서 챗봇
  * 프라이버시 걱정 없는 로컬 AI 어시스턴트
  * 코딩/문서 요약/번역 등 다양한 LLM 활용 앱의 엔진

* **장점**

  * 설치와 사용이 매우 쉽다.
  * 인터넷 연결 없이도 사용 가능하다.
  * 다양한 오픈소스 LLM 지원.

# Ollama로 로컬 LLM 실행

Ollama는 오픈소스 LLM을 다운로드하고 로컬 API 서버로 실행하는 런타임이다. 이 실습에서 `로컬`은 RunPod 내부의 Python 코드와 Ollama 서버가 같은 Pod에서 통신한다는 뜻이다.

모델 자체가 무료로 배포되어도 실행 장비·저장 공간·모델 라이선스 조건은 별도로 확인해야 한다.

공식 자료: [RunPod에서 Ollama 실행](https://docs.runpod.io/tutorials/pods/run-ollama), [Ollama Python 라이브러리](https://github.com/ollama/ollama-python)

## RunPod Pod와 Ollama 서버 준비

[RunPod Ollama 공식 가이드](https://docs.runpod.io/tutorials/pods/run-ollama)와 같은 순서로 새 Pod를 준비한다. Pod을 배포할 때는 PyTorch template을 선택하고 다음 두 설정을 추가한다.

- **Expose HTTP Port**: `11434`
- **Environment Variable**: `OLLAMA_HOST=0.0.0.0`

Pod가 시작되면 **RunPod Web Terminal**에서 다음 명령을 실행한다. `lshw`는 하드웨어 정보 확인에, `zstd`는 압축된 파일 처리에 사용된다. 설치 스크립트가 끝나면 `ollama serve`를 background로 실행하고 log를 `ollama.log`에 저장한다.

```bash
apt update && apt install -y lshw zstd
(curl -fsSL https://ollama.com/install.sh | sh && ollama serve > ollama.log 2>&1) &
```

설치가 완료된 뒤 `ollama run`으로 모델을 다운로드하고 터미널 대화까지 한 번에 확인한다. prompt에 간단한 질문을 입력해 응답을 확인한 후 `/bye`로 종료한다.

```bash
ollama run llama3.2
```

대화 프롬프트에서 질문을 입력한 후 `/bye`를 입력해 종료한다. 다시 shell prompt가 나오면 다운로드된 모델을 확인한다.

```bash
ollama list
```

- `OLLAMA_HOST=0.0.0.0`은 Ollama 서버가 Pod 외부에서 들어오는 요청도 받을 수 있게 binding 주소를 설정한다.
- `ollama run`은 모델이 없으면 먼저 다운로드하고 이어서 대화형 CLI를 연다.
- `ollama list`는 내려받은 모델의 정확한 이름과 tag를 확인한다.

이 노트북의 Python 코드는 같은 Pod의 `http://localhost:11434`에 요청한다. RunPod Proxy URL을 사용한 외부 PyCharm 연결은 배포 노트북에서 다룬다.

### Python 클라이언트 설치

`ollama` 패키지는 실행 중인 Ollama 서버에 `generate()`와 `chat()` 요청을 보내는 공식 Python 클라이언트이다. `%pip`는 현재 Jupyter kernel이 사용하는 Python 환경에 패키지를 설치한다.

In [1]:
%pip install -U ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ollama]2m3/5 [pydantic]
Note: you may need to restart the kernel to use updated packages.


## 단일 prompt로 생성하기

`ollama.generate()`는 문자열 prompt를 받아 `GenerateResponse`를 반환한다. 인자를 별도로 주지 않으면 같은 Pod의 기본 주소 `http://localhost:11434`에 요청한다.

- `model`: `ollama list`에서 확인한 모델 tag이다.
- `prompt`: 모델에 전달할 입력 문자열이다.
- `stream=False`: 완성 응답 한 개를 반환한다.
- `options`: 토큰 수와 샘플링 방식을 지정한다.

In [2]:
import ollama

OLLAMA_MODEL = 'llama3.2'

generation_response = ollama.generate(
    model=OLLAMA_MODEL,
    prompt="Explain AI in simple terms.",
    stream=False, # 생성된 답변을 한 번에 받기
)

# ollama.generate()의 반환 타입 : GenerateResponse
print(generation_response['response'])

Artificial Intelligence (AI) is a way to make machines think and learn like humans do.

**What does "think" mean in the context of AI?**

In the context of AI, "thinking" means that a machine can process information, make decisions, and solve problems on its own. It's like a machine that can learn and improve over time, just like we do.

**How does AI work?**

AI works by using special computer programs that can analyze data, identify patterns, and make predictions. These programs can be trained on large amounts of data, which allows them to learn and improve over time.

**What are some examples of AI?**

Some examples of AI include:

* Virtual assistants like Siri, Alexa, and Google Assistant
* Self-driving cars
* Image recognition software that can identify objects in photos
* Chatbots that can have conversations with customers
* Recommendation systems that suggest products to buy online

**What are the benefits of AI?**

The benefits of AI include:

* Making life easier and more con

### 응답 metadata 읽기

완성 응답에는 생성문뿐 아니라 처리량과 지연 시간도 들어 있다.

- `prompt_eval_count`: 처리한 입력 토큰 수이다.
- `eval_count`: 생성한 출력 토큰 수이다.
- `total_duration`: 모델 로딩과 추론을 포함한 전체 시간이며 나노초 단위이다.

로컬 실행에는 토큰당 API 요금이 없지만 입력 길이와 출력 길이가 늘면 GPU·CPU 사용 시간은 증가한다.

In [4]:
print("input tokens:", generation_response['prompt_eval_count'])
print("output tokens:", generation_response['eval_count'])
print("total duration:", generation_response['total_duration'], "ns")

input tokens: 32
output tokens: 330
total duration: 4980940996 ns


## 생성 옵션 바꾸기

Ollama의 최대 생성 길이는 `max_length`가 아니라 `num_predict`로 지정한다.

- `num_predict`: 최대 출력 토큰 수이다.
- `temperature`: 낮을수록 높은 확률의 토큰을 더 일관되게 고른다.
- `top_p`: 누적 확률 범위 안의 후보만 남기는 nucleus sampling 값이다.
- `seed`: 같은 모델·입력·환경에서 재현성을 높이는 난수 seed이다.

`temperature=0`만으로 모든 환경에서 완전히 같은 문장을 보장하지는 않는다. 비교 실습에서는 `seed`도 함께 고정한다.

## 역할이 있는 Chat 호출

`ollama.chat()`은 문자열 하나가 아니라 message 목록을 받는다.

- `system`: 답변의 역할과 행동 기준이다.
- `user`: 현재 사용자의 요청이다.
- `assistant`: 앞선 모델 응답을 다시 전달할 때 사용한다.

Ollama 서버가 대화 이력을 자동 저장하는 것이 아니다. 이전 대화를 이어 가려면 필요한 message를 다음 요청 목록에 다시 포함해야 한다.

In [5]:
def generate_text(
        prompt:str,
        num_predict: int = 80,
        temperature: float = 0.2,
        top_p: float = 0.9,
        seed: int = 42,


) -> str:

    response = ollama.generate(
        model= OLLAMA_MODEL,
        prompt=prompt,
        stream=False,
        options={
            "num_predict": num_predict,
            "temperature": temperature,
            "top_p": top_p,
            "seed": seed,
        },
    )
    return response['response']

In [6]:
fixed_output = generate_text('Why is the sea salty?')
print(fixed_output)


The sea is salty because of a combination of geological and biological processes that have been occurring over millions of years. Here's a simplified explanation:

1. **Weathering and erosion**: Rainwater and wind can dissolve minerals from rocks and soil, carrying them into rivers and streams. These minerals, such as sodium, chloride, and magnesium, are then transported to the ocean.
2. **Evaporation**:


In [7]:
messages = [
    {"role": "system", "content": "어려운 용어를 피하고 두 문장으로 설명한다. 한글로 대답할 것."},
    {"role": "user", "content": "로컬 LLM의 장점 한 가지와 한계 한 가지를 알려줘."},
]

chat_response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=messages,
    stream=False,
)

print(type(chat_response))

# chat()의 응답 본문은 'response'가 아니고 'message.content'에 있음.
print(chat_response['message'])

<class 'ollama._types.ChatResponse'>
role='assistant' content='로컬 LLM(Large Language Model)의 장점:\n\n1.  **개인 정보 보호**: 로컬 LLM은 중앙서버에存储되는 대신 로컬 컴퓨터에 lưu trữ되므로 개인 정보 보호가 더 좋습니다.\n2.  **속도**: 로컬 LLM은 중앙서버에 비해 속도가 더 빠르기 때문에 더 빠르게 처리할 수 있습니다.\n\n로컬 LLM의 한계:\n\n1.  **데이터량**: 로컬 LLM은中央서버와 compared to large amounts of data가 necessary를 필요로 so larger amounts of data가 필요할 수 있습니다.\n2.  **수학적 모델의 지표**: 로컬 LLM은 중앙서버의 모델을 바탕으로 so 제한된 데이터와 성능이 낮을 수 있습니다.' thinking=None images=None tool_name=None tool_calls=None


## Streaming 응답

`stream=True`이면 완성 응답 하나 대신 여러 `ChatResponse` 조각을 순서대로 반환한다. 각 조각은 토큰 한 개와 정확히 일치하지 않을 수 있으므로 `message.content`를 도착 순서대로 이어 출력한다.

In [8]:
messages = [
    {"role": "user" , "content": "Ollama가 무엇인지 설명해줘"}
]

# stream = True
# -> 응답 조각 (chunk) 중 마지막을 뜻하는 조각이 올 때까지 연결 유지
response_stream = ollama.chat(
    model=OLLAMA_MODEL,
    messages=messages,
    stream=True,
)

for chunk in response_stream:
    print(chunk['message']['content'], end='', flush=True)

Ollama는 K-pop에서 알려진 가수입니다. 오λλ라마는 2013년 한국에서 데뷔했습니다. 오llama는 2014년 4월에 이적하여 2014년 5월에 데뷔했습니다.

오llama는 '어린 시절을 되찾아'라는 메시지를 주는 가수입니다. 오llama는 '어린 시절을 되찾아'라는 곡을 통해 많은 사람들의 사랑을 받았습니다. 오llama는 또한 '어린 시절을 되찾아'라는 곡을 통해 많은 사람들의 사랑을 받았습니다.